# Train and evaluate segmentation model

This notebook runs `train.py` and then evaluates one checkpoint with `evaluate.py` on the validation split.

Check `SPLIT_FILE`, `GEN_ROOT`, and `OUT_DIR` before launching training.

## 1. Optional: mount Google Drive

Run this cell only in Colab if the project or data live in Drive.

In [25]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not running in Colab, or Drive mount was skipped:', exc)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Set project directory

In [26]:
%cd /content/drive/MyDrive/diffusion-segmentation

/content/drive/MyDrive/diffusion-segmentation


In [27]:
from pathlib import Path
import os

PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)

print('Working directory:', Path.cwd())
print('train.py exists:', Path('train.py').exists())
print('evaluate.py exists:', Path('evaluate.py').exists())

Working directory: /content/drive/MyDrive/diffusion-segmentation
train.py exists: True
evaluate.py exists: False


## 3. Install dependencies

In [ ]:
%pip install -q monai nibabel scipy tqdm pandas wandb matplotlib

## 4. Configure paths and check GPU

In [ ]:
import torch

SPLIT_FILE = Path('atlas_train_val.csv')
GEN_ROOT = Path('/scratch/peirong/kxu56/USB/assets/uncond')
OUT_DIR = Path('outputs/colab_train')

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

print('split file:', SPLIT_FILE.resolve(), SPLIT_FILE.exists())
print('gen root:', GEN_ROOT, GEN_ROOT.exists())
print('out dir:', OUT_DIR)

## 5. Run training

For a quick smoke test, reduce workers and use a tiny temporary split file.

In [ ]:
# import subprocess

# CACHE_WORKERS = 4
# LOADER_WORKERS = 4
# GEN_RATIO = 0.0
# GEN_SEED = 40
# SAVE_EVERY = 5
# DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# train_cmd = [
#     'python', 'train.py',
#     '--split-file', str(SPLIT_FILE),
#     '--out-dir', str(OUT_DIR),
#     '--gen-root', str(GEN_ROOT),
#     '--gen-ratio', str(GEN_RATIO),
#     '--gen-seed', str(GEN_SEED),
#     '--cache-workers', str(CACHE_WORKERS),
#     '--loader-workers', str(LOADER_WORKERS),
#     '--device', DEVICE,
#     '--save-every', str(SAVE_EVERY),
#     '--show-progress',
# ]

# print(' '.join(train_cmd))
# result = subprocess.run(train_cmd, text=True)
# if result.returncode != 0:
#     raise RuntimeError(f'train.py failed with return code {result.returncode}')

## 6. Run validation evaluation

In [ ]:
best_checkpoints = sorted(
    OUT_DIR.parent.glob(OUT_DIR.name + '*/unet3d_best_*.pt'),
    key=lambda path: path.stat().st_mtime,
)
if not best_checkpoints:
    raise FileNotFoundError(f'No best checkpoint found under {OUT_DIR.parent}')

EVAL_CHECKPOINT = best_checkpoints[-1]
EVAL_METRICS = Path('eval_results/metrics.json')
EVAL_VIS_DIR = Path('eval_results/visualizations')
VIS_COUNT = 5

eval_cmd = [
    'python', 'evaluate.py',
    '--checkpoint', str(EVAL_CHECKPOINT),
    '--split-file', str(SPLIT_FILE),
    '--metrics-json', str(EVAL_METRICS),
    '--vis-dir', str(EVAL_VIS_DIR),
    '--vis-count', str(VIS_COUNT),
]

print(' '.join(eval_cmd))
result = subprocess.run(eval_cmd, text=True)
if result.returncode != 0:
    raise RuntimeError(f'evaluate.py failed with return code {result.returncode}')

## 7. Inspect outputs

In [ ]:
import json
import pandas as pd

print('Output directories:')
for path in sorted(OUT_DIR.parent.glob(OUT_DIR.name + '*')):
    print(' -', path)

summary_files = sorted(OUT_DIR.parent.glob(OUT_DIR.name + '*/run_summary.json'))
if summary_files:
    latest_summary = summary_files[-1]
    print('\\nLatest summary:', latest_summary)
    print(json.dumps(json.loads(latest_summary.read_text()), indent=2))

if EVAL_METRICS.exists():
    metrics = json.loads(EVAL_METRICS.read_text())
    print('\\nEvaluation metrics:', EVAL_METRICS)
    print('Dice:', metrics.get('dice'))
    print('IoU:', metrics.get('iou'))

registry = OUT_DIR.parent / 'experiment_registry.csv'
if registry.exists():
    display(pd.read_csv(registry).tail())